# Retrieval-Augmented Generation (RAG) Tutorial: Chunking, Embedding, and Retrieval

This notebook demonstrates the core components of Retrieval-Augmented Generation (RAG):
- **Chunking**: Splitting large texts into manageable pieces
- **Embedding**: Converting text chunks into vector representations
- **Retrieval**: Finding relevant chunks based on a query using vector similarity

We'll explore these concepts with examples, starting from simple text and progressing to real-world data like a chapter from a deep learning book.

## Introduction to RAG

Retrieval-Augmented Generation (RAG) is a technique that enhances large language models by retrieving relevant information from a knowledge base before generating responses. This approach improves accuracy, reduces hallucinations, and allows models to access up-to-date information.

The three main components are:

1. **Data Chunking**: Breaking down large documents into smaller, semantically coherent pieces
2. **Embedding**: Converting text chunks into numerical vectors that capture semantic meaning
3. **Retrieval**: Using vector similarity search to find the most relevant chunks for a given query

In this tutorial, we'll implement each component step-by-step and test them on sample data.

## 1. Data Chunking

Chunking is the process of splitting large texts into smaller pieces that can be effectively processed by embedding models and retrieval systems. Good chunking preserves semantic meaning and context while staying within token limits.

### Key Considerations:
- **Chunk Size**: Balance between context preservation and computational efficiency
- **Overlap**: Ensures continuity between chunks, reducing information loss at boundaries
- **Separators**: Use document structure (paragraphs, sentences) for smarter splitting

We'll use LangChain's `RecursiveCharacterTextSplitter` which recursively splits on separators in order of priority.

In [ ]:
import re
from langchain_text_splitters import RecursiveCharacterTextSplitter

def inspect_chunk_overlap_words2(chunks, context_words=5):
    """
    Visualizes chunk boundaries and overlaps to understand how text is split.

    This function helps analyze:
    - How chunks are divided
    - Where overlaps occur
    - Potential context loss at boundaries
    """
    # ANSI color codes for terminal output
    RED = '\033[91m'     # Context before overlap
    GREEN = '\033[92m'   # Overlap region
    BLUE = '\033[94m'    # Context after overlap
    ENDC = '\033[0m'     # Reset color

    print(f"{'--- CHUNK OVERLAP ANALYSIS ---':^100}")
    print("-" * 100)

    def normalize(words):
        """Normalize words by removing punctuation and converting to lowercase."""
        return [re.sub(r'[^\w]', '', w).lower() for w in words]

    def find_word_overlap(a_words, b_words):
        """Find the maximum word overlap between two text segments."""
        a_norm = normalize(a_words)
        b_norm = normalize(b_words)

        max_len = min(len(a_norm), len(b_norm))
        for size in range(max_len, 0, -1):
            if a_norm[-size:] == b_norm[:size]:
                return size
        return 0

    for i in range(len(chunks) - 1):
        a_words = chunks[i].split()
        b_words = chunks[i + 1].split()

        overlap_size = find_word_overlap(a_words, b_words)

        # Extract context windows around the overlap
        overlap_words = a_words[-overlap_size:] if overlap_size > 0 else []
        before_words = a_words[-(overlap_size + context_words):-overlap_size] if overlap_size > 0 else a_words[-context_words:]
        after_words = b_words[overlap_size:overlap_size + context_words]

        before_text = " ".join(before_words)
        overlap_text = " ".join(overlap_words)
        after_text = " ".join(after_words)

        # Display with color coding
        print(
            f"Chunk {i} → {i+1}: "
            f"{RED}{before_text}{ENDC} | "
            f"{GREEN}{overlap_text}{ENDC} | "
            f"{BLUE}{after_text}{ENDC}"
        )
        print()

# Sample technical text for demonstration
example_text = """
The Transformer model relies on Self-Attention mechanisms.
Self-attention allows the model to weigh the importance of different words in a sequence.
This is mathematically achieved through Query, Key, and Value matrices.
By calculating the dot product of Query and Key, we get an attention score.
Positional encodings are necessary because transformers lack the inherent sequential nature of RNNs.
The Adam optimizer adaptively tunes the learning rate, which is why it is preferred for deep learning engineering.
"""

print("=== Chunking Demonstration ===")

# Test different chunk sizes and overlaps
chunk_configs = [
    {"size": 200, "overlap": 20, "desc": "Small chunks with minimal overlap"},
    {"size": 200, "overlap": 100, "desc": "Small chunks with significant overlap"}
]

for config in chunk_configs:
    print(f"\n{config['desc']} (size={config['size']}, overlap={config['overlap']})")
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=config['size'],
        chunk_overlap=config['overlap'],
        separators=["\n\n", "\n", ". ", " ", ""]  # Prioritize structure
    )

    chunks = splitter.split_text(example_text)
    inspect_chunk_overlap_words2(chunks, context_words=4)

## 2. Text Embedding

Embeddings convert text into numerical vectors that capture semantic meaning. Similar texts will have similar vectors, enabling mathematical comparison.

### Key Concepts:
- **Semantic Similarity**: Texts with similar meaning are close in vector space
- **Dimensionality**: Balance between expressiveness and computational cost
- **Normalization**: Ensures fair distance calculations

We'll use SentenceTransformers' `all-MiniLM-L6-v2`, a lightweight model optimized for semantic search.

In [ ]:
import numpy as np
from sentence_transformers import SentenceTransformer
import hnswlib

print("=== Embedding and Retrieval Demonstration ===")

# Configure text splitter for embedding
splitter = RecursiveCharacterTextSplitter(
    chunk_size=120,
    chunk_overlap=20,
    separators=["\n\n", "\n", ". ", " ", ""]
)

# Split the example text into chunks
chunks = splitter.split_text(example_text)
print(f"Created {len(chunks)} chunks from the example text")

# Initialize the embedding model
# all-MiniLM-L6-v2 is a small, fast model good for semantic search
embed_model = SentenceTransformer("all-MiniLM-L6-v2")
print(f"Loaded embedding model with dimension: {embed_model.get_sentence_embedding_dimension()}")

# Generate embeddings for all chunks
chunk_texts = chunks
embeddings = embed_model.encode(chunk_texts, normalize_embeddings=True)
print(f"Generated embeddings with shape: {embeddings.shape}")

# Initialize HNSW index for efficient similarity search
dim = embeddings.shape[1]  # Embedding dimension
num_elements = len(embeddings)

# HNSW (Hierarchical Navigable Small World) provides fast approximate nearest neighbor search
index = hnswlib.Index(space='cosine', dim=dim)
index.init_index(max_elements=num_elements, ef_construction=200, M=16)
index.add_items(embeddings, np.arange(num_elements))
print("Built HNSW index for fast retrieval")

# Test retrieval with a query
query = "How do transformers handle word order?"
query_embedding = embed_model.encode([query], normalize_embeddings=True)

# Find k nearest neighbors
k = 2
labels, distances = index.knn_query(query_embedding, k=k)

print(f"\nQuery: '{query}'")
print(f"Retrieved {k} most similar chunks:")
for i, idx in enumerate(labels[0]):
    similarity_score = 1 - distances[0][i]  # Convert cosine distance to similarity
    print(f"\n--- Chunk {int(idx)} (Similarity: {similarity_score:.3f}) ---")
    print(chunks[int(idx)])

## 3. Vector Retrieval

Retrieval finds the most relevant text chunks for a query by measuring vector similarity. This is the "R" in RAG.

### Key Concepts:
- **Similarity Metrics**: Cosine similarity measures angle between vectors
- **Approximate Search**: HNSW enables fast search in large datasets
- **Ranking**: Results are ordered by relevance score

The retrieval system allows us to find contextually relevant information efficiently.

In [ ]:
# Demonstration: Retrieval on Test Data
# Using the chunks and embeddings from the previous section

print("=== Retrieval Demonstration ===")

# Test queries on the example text
test_queries = [
    "How do transformers handle sequences?",
    "What is self-attention?",
    "Why are positional encodings needed?"
]

for query in test_queries:
    print(f"\nQuery: '{query}'")

    # Embed the query
    query_embedding = embed_model.encode([query], normalize_embeddings=True)

    # Retrieve top 2 most similar chunks
    labels, distances = index.knn_query(query_embedding, k=2)

    for i, idx in enumerate(labels[0]):
        similarity = 1 - distances[0][i]  # Cosine similarity = 1 - cosine distance
        print(f"  Chunk {int(idx)} (Similarity: {similarity:.3f}): {chunks[int(idx)][:100]}...")

print("\nThis shows how the retrieval system finds relevant information based on semantic similarity!")

## 4. Real-World Example: Processing a Book Chapter

Now let's apply the complete RAG pipeline to a real document - Chapter 1 from "Deep Learning with Python" (3rd Edition). This demonstrates how to handle longer, structured markdown documents.

### Considerations for Real Documents:
- **Larger Chunk Sizes**: Preserve more context for complex topics
- **Markdown Structure**: Leverage headers and paragraphs for better splitting
- **Domain-Specific Queries**: Test with questions relevant to the content

In [ ]:
print("=== Real-World RAG Pipeline: Deep Learning Book Chapter ===")

# Load the markdown document
data_path = r"C:\Users\k97ev\Documents\git_repos\dat255-teaching-assistant\chapters_chapter01_what-is-deep-learning.md"
with open(data_path, "r", encoding="utf-8") as f:
    chapter_text = f.read()

print(f"Loaded document with {len(chapter_text)} characters")

# Import required libraries
import numpy as np
from sentence_transformers import SentenceTransformer
import hnswlib

# Configure splitter for longer, structured text
# Larger chunks preserve more context for educational content
splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200,  # Significant overlap for continuity
    separators=["\n\n", "\n", ". ", " ", ""]  # Markdown-aware splitting
)

# Split the chapter into chunks
chunks = splitter.split_text(chapter_text)
print(f"Split into {len(chunks)} chunks")

# Initialize embedding model (reuse for consistency)
embed_model = SentenceTransformer("all-MiniLM-L6-v2")

# Generate embeddings for all chunks
chunk_texts = chunks
embeddings = embed_model.encode(chunk_texts, normalize_embeddings=True)
print(f"Generated embeddings with shape: {embeddings.shape}")

# Build HNSW index
dim = embeddings.shape[1]
num_elements = len(embeddings)
index = hnswlib.Index(space='cosine', dim=dim)
index.init_index(max_elements=num_elements, ef_construction=200, M=16)
index.add_items(embeddings, np.arange(num_elements))
print("Built retrieval index")

# Test with domain-specific queries
test_queries = [
    "What is deep learning?",
    "How does machine learning differ from deep learning?",
    "What are the achievements of deep learning?"
]

for query in test_queries:
    print(f"\n{'='*60}")
    print(f"Query: '{query}'")

    # Embed the query
    query_embedding = embed_model.encode([query], normalize_embeddings=True)

    # Retrieve top 2 most similar chunks
    labels, distances = index.knn_query(query_embedding, k=2)

    for i, idx in enumerate(labels[0]):
        similarity = 1 - distances[0][i]
        print(f"\n--- Retrieved Chunk {int(idx)} (Similarity: {similarity:.3f}) ---")
        # Show first 500 characters of the chunk for readability
        chunk_preview = chunks[int(idx)][:500] + "..." if len(chunks[int(idx)]) > 500 else chunks[int(idx)]
        print(chunk_preview)

print(f"\n{'='*60}")
print("RAG Pipeline Complete!")
print("This demonstrates how to build a retrieval system for educational content.")
print("In a full RAG application, these retrieved chunks would be passed to an LLM for answer generation.")

In [ ]:
# 4. Save index for later use (optional)
# Saving the index lets you reload the vector database quickly without recomputing embeddings.
index_path = "dat255_index.bin"
index.save_index(index_path)
print(f"Index saved to {index_path}")

# Optional: verify load works (uncomment to test)
# loaded_index = hnswlib.Index(space='cosine', dim=dim)
# loaded_index.load_index(index_path)
# print("Loaded index from disk")

## Conclusion

This notebook demonstrated the complete RAG pipeline:

1. **Chunking**: Intelligently splitting documents while preserving structure
2. **Embedding**: Converting text to semantically meaningful vectors
3. **Retrieval**: Efficiently finding relevant information using vector similarity

### Key Takeaways:
- Chunk size and overlap significantly affect retrieval quality
- Markdown structure helps create more coherent chunks
- HNSW enables fast search even with large document collections
- Domain-specific queries work best with relevant training data

### Next Steps:
- Integrate with a language model for generation
- Experiment with different embedding models
- Add metadata and filtering capabilities
- Implement persistent storage for larger knowledge bases

RAG systems like this power many modern AI applications, from chatbots to research assistants!